<a href="https://colab.research.google.com/github/gkmfrombs/isro-land-use-classifier/blob/main/02_Data_Pipelineipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Loading the data**
**Land Cover Classification : Bhuvan Satellite Data**



--------------------------------------------------------------------
--------------------------------------------------------------------

# 🛰️ ISRO Bhuvan Land Cover Classification (Varanasi)

### Dataset Overview
This notebook uses high-resolution 2D satellite imagery of Varanasi, India, sourced from the Indian Space Research Organisation (ISRO). It is pre-processed for pixel-level image segmentation tasks.

**The 5 Target Classes:**
* Vegetation
* Urban Areas
* Forest
* Water Bodies
* Roads

**Data Structure:**
* `train_image` & `test_image`: The raw satellite photos fed into the neural network.
* `train_mask` & `test_mask`: The labeled images where every pixel is color-coded to a target class.
* `class_dict_seg.csv`: Maps the exact RGB colors in the masks to their actual class names.

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("khushiipatni/satellite-image-and-mask")

print("Path to dataset files:", path)

100%|██████████| 7.39M/7.39M [00:01<00:00, 4.16MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/khushiipatni/satellite-image-and-mask/versions/4


The dataset has been successfully downloaded to the following path: `/root/.cache/kagglehub/datasets/khushiipatni/satellite-image-and-mask/versions/4`

##**Preprocessing And Pipline**

In [11]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [12]:
dataset_path = '/root/.cache/kagglehub/datasets/khushiipatni/satellite-image-and-mask/versions/4'
image_dir = os.path.join(dataset_path, 'train_image')
mask_dir = os.path.join(dataset_path, 'train_mask')

In [13]:
# 1. Gather all file paths and sort them alphabetically
train_image_paths = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))])
train_mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir) if f.endswith(('.jpg', '.png'))])

print(f"Total Images Found: {len(train_image_paths)}")
print(f"Total Masks Found: {len(train_mask_paths)}")

# Safety check: Ensure we have the exact same number of images and masks
assert len(train_image_paths) == len(train_mask_paths), "Mismatch! The number of images and masks do not match."
print("Safety check passed! Images and masks are aligned.")

Total Images Found: 60
Total Masks Found: 60
Safety check passed! Images and masks are aligned.


In [14]:
# Target size for the Neural Network
IMG_SIZE = 256

# Exact mapping from your dataset
class_mapping = {
    'urban': (0, [0, 255, 255]),
    'water': (1, [0, 0, 255]),
    'forest': (2, [0, 255, 0]),
    'argiculture': (3, [255, 255, 0]),
    'road': (4, [255, 0, 255])
}

def preprocess_mask(mask_image):
    # Resize mask without blending colors (INTER_NEAREST)
    resized = cv2.resize(mask_image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    # Create empty single-channel array
    label_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    # Swap RGB colors for Class IDs
    for class_info in class_mapping.values():
        class_id = class_info[0]
        color = class_info[1]
        matches = np.all(resized == color, axis=-1)
        label_mask[matches] = class_id

    return np.expand_dims(label_mask, axis=-1)

print("Preprocessing rules defined successfully!")

Preprocessing rules defined successfully!


In [15]:
from sklearn.model_selection import train_test_split

# Split the 60 file paths:
# 50 images for Training (80%), 10 images for Validation (20%)
train_x, val_x, train_y, val_y = train_test_split(
    train_image_paths,
    train_mask_paths,
    test_size=0.2,
    random_state=42 # This number ensures the split is the same every time you run it
)

print(f"Training on: {len(train_x)} images")
print(f"Validating on: {len(val_x)} images")

Training on: 48 images
Validating on: 12 images


In [18]:
import tensorflow as tf

# 1. The Worker: Processes a single image/mask pair
def process_path(img_path, mask_path):
    # TensorFlow passes file paths as byte strings, so we decode them to standard text
    img_path = img_path.decode('utf-8')
    mask_path = mask_path.decode('utf-8')

    # Process the Satellite Image
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0  # Normalize: scales pixels from 0-255 down to 0.0-1.0

    # Process the Mask
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
    processed_mask = preprocess_mask(mask)

    # Return both as float32 decimal numbers (the standard for neural networks)
    return img.astype(np.float32), processed_mask.astype(np.float32)

# 2. The Translator: Wraps our Python worker so TensorFlow can use it
def tf_parse(img_path, mask_path):
    # tf.numpy_function runs our python code inside the fast TensorFlow graph
    img, mask = tf.numpy_function(process_path, [img_path, mask_path], [tf.float32, tf.float32])

    # We must explicitly tell TensorFlow the final shapes so it doesn't get confused
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])

    return img, mask

# 3. The Conveyor Belt: Building the actual pipeline
BATCH_SIZE = 8

# IMPORTANT: We use train_x and train_y here (from our split),
# NOT the original train_image_paths list!
train_dataset = tf.data.Dataset.from_tensor_slices((train_x, train_y))
train_dataset = train_dataset.map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=50).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Also create the Validation Pipeline here
val_dataset = tf.data.Dataset.from_tensor_slices((val_x, val_y))
val_dataset = val_dataset.map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Data Pipelines (Train & Val) successfully built!")
for image_batch,mask_batch in train_dataset.take(1):
  print(f"Batch Image Shape: {image_batch.shape}")
  print(f"Batch Mask Shape: {mask_batch.shape}")

Data Pipelines (Train & Val) successfully built!
Batch Image Shape: (8, 256, 256, 3)
Batch Mask Shape: (8, 256, 256, 1)


------------------------------------------------------------------



### 1. Summary

### **Project Phase 2: Data Pipeline & Validation Strategy**

**Situation:** With the data explored and preprocessing logic verified, the project required a scalable way to feed 60 high-resolution satellite images into a neural network without crashing system memory.

**Task:** The goal was to build a robust data pipeline that handles automated resizing, normalization, and class mapping, while also establishing a "Validation Set" to monitor the model's learning progress.

**Action:**

1. **Data Splitting:** Used `train_test_split` to divide the 60 images into a **Training Set (80%)** for learning and a **Validation Set (20%)** for "practice quizzes" during training.
2. **Standardization:** Implemented a normalization step to scale pixel values from [0-255] to **[0.0-1.0]**, making the data mathematically compatible with Deep Learning optimizers.
3. **TensorFlow Pipeline:** Created a `tf.data.Dataset` "conveyor belt" that utilizes `prefetching` and `batching`. This ensures the GPU stays busy by preparing the next batch of images in the background while the model is learning from the current one.
4. **Error Handling:** Incorporated a `tf_parse` bridge to safely run Python-based OpenCV functions within TensorFlow's high-speed execution environment.

**Result:**

* Established two high-performance data streams: `train_dataset` and `val_dataset`.
* Successfully verified batch shapes of **(8, 256, 256, 3)** for images and **(8, 256, 256, 1)** for masks.
* The project is now architecture-ready, with data flowing efficiently from storage to the GPU-accelerated training environment.

---

